# Sprint 2 — Component 2 TB Head Training

Trains the TorchXRayVision routing head + TB classification head.

**What changed in Sprint 2:**
- Separate LR groups: routing head at `lr`, TB head at `lr × 5.0` (fresh Linear needs faster LR)
- `pos_weight = n_neg / n_pos` in BCE loss to handle TB/normal class imbalance
- Early stopping now monitors val TB AUROC (not combined loss)
- Per-epoch AUROC printed — you can see the head actually learning
- epochs=30, patience=10, tb_head_weight=2.0
- Backbone runs once per batch (shared pooled features for both heads)

**Datasets to attach:**
- `iahmedhabib/montgomery`
- `iahmedhabib/shehzhenn`
- `usmanshams/tbx-11`

NIH is not needed — it has no TB labels and is excluded from training.

In [4]:
# Cell 1 — Clone repo and install dependencies
# torchxrayvision is NOT pre-installed on Kaggle — must be installed explicitly.
import subprocess, sys, os

REPO_DIR = '/kaggle/working/dl-project'

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/mabdullahi7780/dl-project-codebase.git', REPO_DIR],
        check=True
    )
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# Install all deps from requirements.txt (now includes torchxrayvision and scikit-learn)
!pip install -q -r requirements.txt

# Explicit install as safety net — torchxrayvision is critical, mock backend = AUROC stuck at 0.5
!pip install -q torchxrayvision scikit-learn

print('Dependencies installed.')

Already up to date.
Working dir: /kaggle/working/dl-project
Dependencies installed.


In [5]:
# Cell 2 — Verify dataset paths
from pathlib import Path

MONTGOMERY_ROOT = '/kaggle/input/datasets/iahmedhabib/montgomery/montgomery'
SHENZHEN_ROOT   = '/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen'
TBX11K_ROOT     = '/kaggle/input/datasets/usmanshams/tbx-11/TBX11K'

def _show(name, path, check_subdirs=None):
    p = Path(path) if path else None
    if p and p.exists():
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'  {name}: OK  ({n} files)  {path}')
        if check_subdirs:
            for sub in check_subdirs:
                sp = p / sub
                print(f'    [{"OK" if sp.exists() else "MISSING"}] {sub}/')
    else:
        print(f'  {name}: NOT FOUND — {path}')
        parent = Path(path).parent if path else None
        if parent and parent.exists():
            print(f'    Parent {parent} contains:')
            for child in sorted(parent.iterdir())[:8]:
                print(f'      {child.name}/')

print('Dataset roots:')
_show('Montgomery', MONTGOMERY_ROOT, ['MontgomerySet/CXR_png', 'CXR_png'])
_show('Shenzhen',   SHENZHEN_ROOT,   ['CXR_png', 'images', 'images/images'])
_show('TBX11K',     TBX11K_ROOT,     ['imgs/tb', 'imgs/health', 'lists'])

shen = Path(SHENZHEN_ROOT)
for sub in ('CXR_png', 'images/images', 'images'):
    p = shen / sub
    if p.exists():
        imgs = list(p.glob('*.png')) + list(p.glob('*.jpg'))
        print(f'  Shenzhen images found at {sub}/: {len(imgs)}')
        break

Dataset roots:
  Montgomery: OK  (555 files)  /kaggle/input/datasets/iahmedhabib/montgomery/montgomery
    [MISSING] MontgomerySet/CXR_png/
    [OK] CXR_png/
  Shenzhen: OK  (1795 files)  /kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen
    [MISSING] CXR_png/
    [OK] images/
    [OK] images/images/
  TBX11K: OK  (13101 files)  /kaggle/input/datasets/usmanshams/tbx-11/TBX11K
    [OK] imgs/tb/
    [OK] imgs/health/
    [OK] lists/
  Shenzhen images found at images/images/: 662


In [6]:
# Cell 3 — Verify TorchXRayVision is installed and real backend will be used
# This MUST print 'xrv' — if it prints 'mock' the training is broken.
import sys
sys.path.insert(0, '/kaggle/working/dl-project')

try:
    import torchxrayvision as xrv
    print(f'torchxrayvision version: {xrv.__version__}')
    print('Backend will be: XRV (real DenseNet) — OK')
except ImportError:
    print('ERROR: torchxrayvision is NOT installed.')
    print('Re-run Cell 1 or run: !pip install torchxrayvision')
    raise

# Quick sanity check: load model and verify backend
import torch
from src.components.component2_txv import Component2SoftDomainContext
model = Component2SoftDomainContext(backend='auto', weights='densenet121-res224-all')
print(f'model.active_backend = {model.active_backend!r}')
assert model.active_backend == 'xrv', f'Expected xrv, got {model.active_backend}'

# Check that features are non-zero on a dummy input
x_dummy = torch.zeros(1, 1, 224, 224)
with torch.no_grad():
    feat = model.txv_model.features(x_dummy)
    pooled = torch.nn.functional.adaptive_avg_pool2d(feat, (1,1)).view(1, -1)
print(f'Pooled feature norm on dummy input: {pooled.norm().item():.4f}')
print('(non-zero means real DenseNet is running — if zero, something is wrong)')
del model

torchxrayvision version: 1.4.0
Backend will be: XRV (real DenseNet) — OK
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
model.active_backend = 'xrv'
Pooled feature norm on dummy input: 15.6612
(non-zero means real DenseNet is running — if zero, something is wrong)


In [7]:
# Cell 4 — Set environment variables
import os

SAVE_DIR = '/kaggle/working/component2_runs'

os.environ['MONTGOMERY_ROOT']     = MONTGOMERY_ROOT
os.environ['SHENZHEN_ROOT']       = SHENZHEN_ROOT
os.environ['TBX11K_ROOT']         = TBX11K_ROOT
os.environ['COMPONENT2_SAVE_DIR'] = SAVE_DIR

os.makedirs(SAVE_DIR, exist_ok=True)

print('Environment:')
for k in ('MONTGOMERY_ROOT', 'SHENZHEN_ROOT', 'TBX11K_ROOT', 'COMPONENT2_SAVE_DIR'):
    print(f'  {k} = {os.environ[k]}')
print()
print('Starting training — watch ValAUROC climb above 0.5...')
print('=' * 60)

Environment:
  MONTGOMERY_ROOT = /kaggle/input/datasets/iahmedhabib/montgomery/montgomery
  SHENZHEN_ROOT = /kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen
  TBX11K_ROOT = /kaggle/input/datasets/usmanshams/tbx-11/TBX11K
  COMPONENT2_SAVE_DIR = /kaggle/working/component2_runs

Starting training — watch ValAUROC climb above 0.5...


In [8]:
# Cell 5 — Run training
!python -m src.training.train_component2_txv \
    --config configs/component2_txv.kaggle.yaml \
    --paths configs/paths.yaml \
    2>&1 | tee /kaggle/working/component2_training_log.txt

INFO: NIH CXR14 root not found at /Volumes/Toshiba HDR/TB_DATA/raw/nih_cxr14 — skipping (sampling weight=1.0).
Backend: xrv | Device: cuda
TB label counts — pos: 944, neg: 3374 | pos_weight: 3.57

  Ep  TrainLoss   ValLoss  ValAUROC                     
----------------------------------------------------------
   1     4.1777    4.1467    0.9608 <-- saved
   2     3.9642    4.0920    0.9685 <-- saved
   3     3.8997    4.0058    0.9707 <-- saved
   4     3.8737    4.0193    0.9737 <-- saved
   5     3.8673    3.9616    0.9760 <-- saved
   6     3.8365    3.9629    0.9768 <-- saved
   7     3.8002    3.9336    0.9782 <-- saved
   8     3.8236    3.9524    0.9786 <-- saved
   9     3.8293    3.9704    0.9804 <-- saved
  10     3.8301    3.9184    0.9795 (patience 1/10)
  11     3.7626    3.9228    0.9799 (patience 2/10)
  12     3.7825    3.9480    0.9813 <-- saved
  13     3.7881    4.0481    0.9811 (patience 1/10)
  14     3.7450    3.8795    0.9821 <-- saved
  15     3.7659    3.9286

In [9]:
# Cell 6 — Verify checkpoint
import torch
from pathlib import Path

ckpt_path = Path('/kaggle/working/component2_runs/component2_routing_head.pt')

if ckpt_path.exists():
    print(f'Checkpoint: {ckpt_path} ({ckpt_path.stat().st_size / 1e6:.1f} MB)')
    state = torch.load(ckpt_path, map_location='cpu')
    print(f'  Routing head keys: {len([k for k in state if k != "tb_head"])}')
    print(f'  TB head saved: {"tb_head" in state and isinstance(state["tb_head"], dict)}')
else:
    print('ERROR: checkpoint not found at', ckpt_path)

# Parse key lines from log
log_path = Path('/kaggle/working/component2_training_log.txt')
if log_path.exists():
    print()
    print('Training summary:')
    print('-' * 60)
    with open(log_path) as f:
        for line in f:
            line = line.rstrip()
            if any(x in line for x in ['Ep', '---', 'pos:', 'Backend', 'AUROC', 'complete', 'Early', 'Best', 'saved']):
                print(line)

Checkpoint: /kaggle/working/component2_runs/component2_routing_head.pt (1.3 MB)
  Routing head keys: 4
  TB head saved: True

Training summary:
------------------------------------------------------------
Backend: xrv | Device: cuda
TB label counts — pos: 944, neg: 3374 | pos_weight: 3.57
  Ep  TrainLoss   ValLoss  ValAUROC
----------------------------------------------------------
   1     4.1777    4.1467    0.9608 <-- saved
   2     3.9642    4.0920    0.9685 <-- saved
   3     3.8997    4.0058    0.9707 <-- saved
   4     3.8737    4.0193    0.9737 <-- saved
   5     3.8673    3.9616    0.9760 <-- saved
   6     3.8365    3.9629    0.9768 <-- saved
   7     3.8002    3.9336    0.9782 <-- saved
   8     3.8236    3.9524    0.9786 <-- saved
   9     3.8293    3.9704    0.9804 <-- saved
  12     3.7825    3.9480    0.9813 <-- saved
  14     3.7450    3.8795    0.9821 <-- saved
  15     3.7659    3.9286    0.9834 <-- saved
  23     3.7449    3.9015    0.9841 <-- saved
  24     3.7068  

In [10]:
# Cell 7 — Per-dataset val AUROC breakdown
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score
from collections import defaultdict
from torch.utils.data import DataLoader

from src.components.component2_txv import Component2SoftDomainContext
from src.training.train_component2_txv import (
    Component2DomainDataset, collate_component2_batch, stratified_split
)
from src.training.train_component1_dann import build_component1_manifest, load_yaml_config

CKPT = '/kaggle/working/component2_runs/component2_routing_head.pt'

if not Path(CKPT).exists():
    print('Checkpoint not found — run Cell 5 first.')
else:
    config = load_yaml_config('configs/component2_txv.kaggle.yaml')['component2_txv']
    samples = build_component1_manifest()
    excluded = {d.lower() for d in config['data'].get('exclude_datasets', [])}
    samples = [s for s in samples if s.dataset_id not in excluded]
    _, val_samples = stratified_split(samples, val_ratio=config['data']['val_split'], seed=1337)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = Component2SoftDomainContext(backend='auto', weights='densenet121-res224-all').to(device)
    model.load_trained_routing_head(CKPT)
    model.eval()
    print(f'Loaded checkpoint | backend={model.active_backend}')

    val_loader = DataLoader(
        Component2DomainDataset(val_samples),
        batch_size=32, shuffle=False, num_workers=2,
        collate_fn=collate_component2_batch,
    )

    by_dataset = defaultdict(lambda: {'logits': [], 'labels': []})

    with torch.no_grad():
        for batch in val_loader:
            x = batch['x_224'].to(device)
            labels = batch['tb_label']
            logits = model.forward_tb_logit(x).squeeze(1).cpu()
            for i, ds in enumerate(batch['dataset_id']):
                by_dataset[ds]['logits'].append(logits[i].item())
                by_dataset[ds]['labels'].append(labels[i].item())

    print(f'\n{"Dataset":<14} {"N_val":>6} {"N_TB":>6} {"AUROC":>8}')
    print('-' * 38)
    for ds, data in sorted(by_dataset.items()):
        lbls = [l for l in data['labels'] if l != -1]
        lgs  = [data['logits'][i] for i, l in enumerate(data['labels']) if l != -1]
        n_tb = sum(1 for l in lbls if l == 1)
        if len(set(lbls)) >= 2:
            import numpy as np
            auc = roc_auc_score(lbls, torch.sigmoid(torch.tensor(lgs)).numpy())
            print(f'{ds:<14} {len(lbls):>6} {n_tb:>6} {auc:>8.4f}')
        else:
            print(f'{ds:<14} {len(lbls):>6} {n_tb:>6} {"N/A":>8}')

INFO: NIH CXR14 root not found at /Volumes/Toshiba HDR/TB_DATA/raw/nih_cxr14 — skipping (sampling weight=1.0).
Loaded checkpoint | backend=xrv

Dataset         N_val   N_TB    AUROC
--------------------------------------
montgomery         28     18   0.5722
shenzhen          133     65   0.8604
tbx11k            921    167   0.9828
